# Synthetic valley with seawater intrusion

This notebook adds the Seawater Intrusion (**SWI**) Package to the coastal synthetic-valley base model, runs it, maps where saltwater lies beneath the valley, and asks whether the pumping wells near the coast draw in seawater. The companion notebook [`synthetic-valley-a-base`](synthetic-valley-a-base.ipynb) runs the same valley without SWI.

> **Model reference.** The synthetic valley model is described in Hughes, J.D., Langevin, C.D., Paulinski, S.R., Larsen, J.D., and Brakenhoff, D., 2024, FloPy workflows for creating structured and unstructured MODFLOW models: Groundwater, v. 62, no. 1, p. 124–139, https://doi.org/10.1111/gwat.13327.

> **Preview package.** SWI is not yet part of a MODFLOW 6 release. The class MODFLOW 6 includes it; see the [README](../../README.md).

## What this notebook covers

By the end of this notebook you will be able to:

- combine the model layers that represent one aquifer, and check that the result matches the original model,
- represent the sea for SWI with an equivalent freshwater head,
- add the SWI Package with `flopy.mf6.ModflowGwfswi()` and let wells cut back their pumping with the Well Package `AUTO_FLOW_REDUCE` option,
- map the interface between freshwater and saltwater and draw it in cross section,
- tell from the well output whether a well is drawing in seawater,
- compare the water budget with and without SWI,
- couple a freshwater and a saltwater model with the SWI-SWI exchange to simulate how fast the saltwater moves, and
- explore the interface through time with an interactive figure.

## Seawater intrusion with SWI

Beneath a coastal aquifer, fresh groundwater floats on denser seawater. SWI treats the two as separate fluids that meet at a sharp **interface**. Each cell holds a freshwater zone above the interface and a saltwater zone below it, and freshwater flows only through the freshwater zone. The interface can lie anywhere within a cell, so the model does not need thin layers to follow it.

This notebook uses SWI in **single-fluid** mode. MODFLOW simulates the freshwater only; the saltwater is assumed to stand still, with its head at sea level. The interface elevation $\zeta$ then follows from the freshwater head $h_f$ and the saltwater head $h_s$:

$$\zeta = -\frac{1}{\delta}\,h_f + \frac{1+\delta}{\delta}\,h_s ,$$

where $\delta$ is the relative density difference, $(\rho_s - \rho_f)/\rho_f$, which is 0.025 for seawater. With $h_s = 0$ this is the Ghyben–Herzberg relation: the interface lies 40 m below sea level for every meter the freshwater head stands above it.

SWI needs the Newton formulation and a specific yield in the Storage (**STO**) Package. The base model already has both.

## The model

The synthetic valley drains south to the sea. The model uses a structured grid of 40 rows and 25 columns of 152.4 meter (m) cells. The base model has five layers: layers 1 and 2 are the upper aquifer, layer 3 is a clay present only in part of the valley, and layers 4 and 5 are the lower aquifer. The stresses are simple packages: recharge (**RCH**), a lake of high hydraulic conductivity cells with evapotranspiration (**EVT**), a stream of River (**RIV**) cells, two pumping wells (**WEL**) that start in the second stress period, a proposed well (**WEL**) that starts in the twelfth, and the sea, a General-Head Boundary (**GHB**) along the southern edge.

Because SWI places the interface within a cell, this notebook combines the two layers of each aquifer into one, which gives three layers:

- layer 1 is the **upper aquifer** (base layers 1 and 2),
- layer 2 is the **clay**, with vertical pass-through cells (`idomain` of -1) where the clay is absent, and
- layer 3 is the **lower aquifer** (base layers 4 and 5).

The first stress period is a 16,071-day **steady-state** period, which gives the flow system before pumping. Twenty one-year transient periods follow. Every input is in meters and days.

## Imports

Import FloPy, Matplotlib, NumPy, and pandas. `sv_notebook_helpers` holds the location of the base model and the function that builds the three-layer model.

In [ ]:
%matplotlib inline
import pathlib as pl

import flopy
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sv_notebook_helpers import BASE_MODEL, build_three_layer

width = 17.15 / 2.54  # figure width, inches

## Build the three-layer model

`build_three_layer()` reads the base model, combines its layers, writes the three-layer model to a new workspace, and loads it with `flopy.mf6.MFSimulation.load()`. Each combined layer takes the properties of its upper layer, which in the base model are the same for both layers of an aquifer. The conductances of the coastal GHB cells and the rates of the wells that are combined are added together. The lake cells of base layer 1 now reach the bottom of the upper aquifer.

Build it in its own workspace, which `pathlib.Path` names, and list its packages. This copy runs without SWI, first as a check against the base model and later to show what SWI changes in the water budget.

In [ ]:
name = "sv"
fresh_ws = pl.Path("models/synthetic-valley-base-3layer")

fresh = build_three_layer(fresh_ws)
fresh.get_model(name).get_package_list()

## Check the three-layer model against the base model

Run the base model and the three-layer model, both without SWI, and compare their heads at the end of the steady-state period and at the end of the simulation. The water table is layer 1 in both models; the lower aquifer is base layer 4 and layer 3 of the three-layer model. `run_simulation()` returns `True` when the run ends normally.

In [ ]:
base = flopy.mf6.MFSimulation.load(sim_ws=BASE_MODEL, verbosity_level=0)
base.set_sim_path("models/synthetic-valley-base-5layer")
base.write_simulation(silent=True)
success, buff = base.run_simulation(silent=True)
assert success, "MODFLOW 6 did not terminate normally"

success, buff = fresh.run_simulation(silent=True)
assert success, "MODFLOW 6 did not terminate normally"

h5 = base.get_model(name).output.head()
h3 = fresh.get_model(name).output.head()
for label, idx in (("steady state", 0), ("after 20 years", -1)):
    a, b = h5.get_data(idx=idx), h3.get_data(idx=idx)
    for aquifer, k5, k3 in (("water table", 0, 0), ("lower aquifer", 3, 2)):
        d = b[k3] - a[k5]
        print(
            f"{label}, {aquifer}: three-layer minus base head, "
            f"mean {d.mean():.3f} m, largest {np.abs(d).max():.3f} m"
        )

Map the water table of each model at the end of the steady-state period, and the three-layer water table minus the base model's. No cell of layer 1 goes dry in either model, so the head in layer 1 is the water table everywhere. The difference map uses a color scale centered on zero: red where the three-layer water table is higher, blue where it is lower.

In [ ]:
wt5 = h5.get_data(idx=0)[0]
wt3 = h3.get_data(idx=0)[0]
diff = wt3 - wt5
vmin, vmax = np.floor(min(wt5.min(), wt3.min())), np.ceil(max(wt5.max(), wt3.max()))
levels = np.arange(vmin, vmax + 1.0, 1.0)
dmax = np.abs(diff).max()
panels = {
    "A": (base.get_model(name), wt5, "Base model (5 layers)"),
    "B": (fresh.get_model(name), wt3, "Three-layer model"),
}

with flopy.plot.styles.USGSMap():
    fig, axd = plt.subplot_mosaic(
        [["A", "B", "C"]], figsize=(width, 0.62 * width), layout="constrained"
    )
    for letter, (model, wt, heading) in panels.items():
        ax = axd[letter]
        mm = flopy.plot.PlotMapView(model=model, ax=ax, layer=0)
        pa = mm.plot_array(wt, vmin=vmin, vmax=vmax)
        cs = mm.contour_array(wt, levels=levels, colors="black", linewidths=0.5)
        ax.clabel(cs, fmt="%g", fontsize=6)
        mm.plot_bc(package=model.riv, color="cyan")
        ax.set_aspect("equal")
        ax.set_xlabel("x (m)")
        flopy.plot.styles.heading(ax=ax, letter=letter, heading=heading)
    axd["A"].set_ylabel("y (m)")
    axd["B"].set_yticklabels([])
    fig.colorbar(
        pa,
        ax=[axd["A"], axd["B"]],
        orientation="horizontal",
        shrink=0.6,
        label="Water table (m)",
    )

    ax = axd["C"]
    mm = flopy.plot.PlotMapView(model=fresh.get_model(name), ax=ax, layer=0)
    pd_ = mm.plot_array(diff, cmap="RdBu_r", vmin=-dmax, vmax=dmax)
    mm.plot_bc(package=fresh.get_model(name).riv, color="0.4")
    ax.set_aspect("equal")
    ax.set_xlabel("x (m)")
    ax.set_yticklabels([])
    flopy.plot.styles.heading(ax=ax, letter="C", heading="Three-layer minus base")
    fig.colorbar(
        pd_, ax=ax, orientation="horizontal", shrink=0.9, label="Difference (m)"
    )
plt.show()

**What to look for.** In both aquifers the three-layer heads are on average within 0.03 m of the base model's heads, and no cell differs by more than 0.2 m, before or after pumping. One layer per aquifer is enough for this valley. The two water-table maps are hard to tell apart (A and B). The difference map (C) shows the three-layer water table 0.02 m higher on average at steady state, most of all, by up to 0.07 m, near the head of the stream, and 0.023 m higher across the whole lake, which stays flat in both models.

## Build a second copy for SWI

The three-layer model matches the base model, so build a second copy of it in a new workspace and add SWI to that one. `sim` is the simulation and `gwf` its groundwater flow model; the cells below change them before the run. The copy without SWI stays as it is for the budget comparison later.

In [ ]:
swi_ws = pl.Path("models/synthetic-valley-swi-1fluid")

sim = build_three_layer(swi_ws)
gwf = sim.get_model(name)

## Represent the sea for SWI

Two changes make the coastal boundary work with SWI.

**An equivalent freshwater head.** MODFLOW simulates freshwater heads, but the sea pushes on the aquifer with the weight of seawater, which is 2.5 percent heavier. At a depth $d$ below sea level the seawater pressure equals that of a freshwater column whose head is

$$h_f = h_s + \delta\, d ,$$

so a GHB cell centered 30 m below sea level needs a head of 0.75 m, not 0. The GHB carries the elevation of the middle of each cell below sea level as its `elevation` auxiliary variable, so the head is one line of NumPy.

**The sea on the upper aquifer only.** A GHB applies one head to a whole cell face. Where most of a coastal cell in the lower aquifer is saltwater, the freshwater head in the cell is lower than the equivalent freshwater head of the sea at the middle of the cell, and the GHB would push freshwater into the aquifer at depth: water that does not exist. In single-fluid mode the saltwater is not simulated, so it needs no boundary. Remove the lower aquifer GHB cells; the freshwater of the lower aquifer then leaves by rising into the upper aquifer near the coast.

Read the GHB data for the first stress period, which applies to every later period, keep the layer 1 cells, set their heads, and put the data back with `set_data()`.

In [ ]:
delta = 0.025  # relative density difference of seawater
sea_level = 0.0  # m

ghb = gwf.ghb.stress_period_data.get_data(0)
ghb = ghb[[cellid[0] == 0 for cellid in ghb["cellid"]]]
ghb["bhead"] = sea_level + delta * (sea_level - ghb["elevation"])
gwf.ghb.stress_period_data.set_data({0: ghb})
print(
    f"{len(ghb)} sea cells in layer 1, equivalent freshwater head "
    f"{ghb['bhead'].min():.3f} m"
)

## Add the SWI Package

`flopy.mf6.ModflowGwfswi()` adds SWI. `relative_density_difference` is $\delta$, and `saltwater_head` defaults to 0 m, which is sea level here. `zeta_filerecord` saves the interface elevation of every cell to a binary file, in the same form as the heads.

In [ ]:
swi = flopy.mf6.ModflowGwfswi(
    gwf,
    relative_density_difference=delta,
    zeta_filerecord=f"{name}.zta",
)

## Let the wells cut back when they reach saltwater

In single-fluid mode a well can only pump the freshwater above the interface. If pumping draws the interface up to the well, the freshwater zone of the well cell runs out, and the model cannot supply the rate: the simulation fails. The Well Package `AUTO_FLOW_REDUCE` option prevents that. When the freshwater zone of a well cell becomes thinner than the given fraction of the cell thickness, here 0.1, the pumping rate is reduced smoothly toward zero. A reduced rate is the model saying that this well would be drawing in seawater.

Set the option on both well packages, and save the reductions to a comma-separated values (CSV) file with `afrcsv_filerecord`. The file lists a well only in the stress periods in which its rate was reduced.

In [ ]:
for pname in ("pwell", "prediction"):
    wel = gwf.get_package(pname)
    wel.auto_flow_reduce = 0.1
    wel.afrcsv_filerecord = f"{name}.{pname}.afr.csv"

## Monthly time steps

Pumping can draw the interface up to a well within months, so one time step per year is too coarse to follow it. Split each one-year transient period into 12 time steps of about a month. The period data of the Time Discretization (**TDIS**) Package gives the length, number of time steps, and time-step multiplier of each period; the steady-state period keeps its single time step.

In [ ]:
perioddata = sim.tdis.perioddata.array.tolist()
sim.tdis.perioddata.set_data(
    [perioddata[0]] + [(perlen, 12, 1.0) for perlen, _, _ in perioddata[1:]]
)

## Run the model

Write the input files with `write_simulation()` and run MODFLOW 6 with `run_simulation()`.

In [ ]:
sim.write_simulation(silent=True)
success, buff = sim.run_simulation(silent=True)
assert success, "MODFLOW 6 did not terminate normally"

## The interface in the lower aquifer

Read the interface elevations from the zeta file with `flopy.utils.HeadFile()`, using `text="zeta"`. Index 0 is the end of the steady-state period, before pumping, and index -1 is the end of the simulation, after 20 years of pumping. A cell holds saltwater where the interface is above its bottom, and is all saltwater where the interface is above its top. Where a cell holds no saltwater, the file gives its bottom.

The upper aquifer holds saltwater only in two cells beneath the mouth of the stream, before and after pumping, where the streambed is below sea level and the heads are within 0.11 m of it. The maps show the lower aquifer (layer 3). The wells are the `reilly` and `vc` wells, which pump from the start of the second stress period, and the proposed well, which starts in the twelfth.

In [ ]:
zeta_file = flopy.utils.HeadFile(swi_ws / f"{name}.zta", text="zeta")
zeta = {"before": zeta_file.get_data(idx=0), "after": zeta_file.get_data(idx=-1)}
head_file = gwf.output.head()
head = {"before": head_file.get_data(idx=0), "after": head_file.get_data(idx=-1)}

top = gwf.dis.top.array
botm = gwf.dis.botm.array
idomain = gwf.dis.idomain.array
tops = np.vstack([top[np.newaxis], botm[:-1]])
extent = gwf.modelgrid.extent

# well name: (row, column), counted from 0
wells = {"reilly": (5, 14), "vc": (32, 5), "proposed": (34, 15)}
for when, z in zeta.items():
    salt = z > botm
    print(
        f"{when} pumping: saltwater in {salt[0].sum()} upper-aquifer cells "
        f"and {salt[2].sum()} lower-aquifer cells"
    )
    for well, (i, j) in wells.items():
        where = (
            "no saltwater"
            if z[2, i, j] <= botm[2, i, j]
            else f"interface at {z[2, i, j]:.1f} m"
        )
        print(
            f"  {well}: {where}; lower aquifer "
            f"{tops[2, i, j]:.1f} to {botm[2, i, j]:.1f} m"
        )

Map the interface elevation in the lower aquifer before and after pumping. `flopy.plot.PlotMapView()` draws each panel, and `plot_array()` colors only the cells that hold saltwater. The black line is the **toe**, the inland limit of the saltwater, and the dashed lines are the cross sections drawn next.

In [ ]:
k = 2  # lower aquifer
vmin, vmax = np.floor(botm[k].min()), np.ceil(tops[k].max())
sections = {"vc": 5, "proposed": 15}  # well: column, counted from 0
panels = {
    "A": ("before", "Before pumping"),
    "B": ("after", "After 20 years of pumping"),
}

with flopy.plot.styles.USGSMap():
    fig, axd = plt.subplot_mosaic(
        [list(panels)], figsize=(width, 0.75 * width), layout="constrained"
    )
    for letter, (when, heading) in panels.items():
        ax = axd[letter]
        z = zeta[when][k]
        salt = z > botm[k]
        mm = flopy.plot.PlotMapView(model=gwf, ax=ax, layer=k)
        pa = mm.plot_array(
            np.ma.masked_where(~salt, np.minimum(z, tops[k])), vmin=vmin, vmax=vmax
        )
        mm.contour_array(salt.astype(float), levels=[0.5], colors="black")
        mm.plot_bc(package=gwf.ghb, color="purple", kper=0)
        for well, (i, j) in wells.items():
            x, y = gwf.modelgrid.xcellcenters[i, j], gwf.modelgrid.ycellcenters[i, j]
            ax.plot(x, y, "o", ms=5, mfc="white", mec="black")
            ax.annotate(well, (x, y), xytext=(4, 4), textcoords="offset points")
        for well, j in sections.items():
            x = gwf.modelgrid.xcellcenters[0, j]
            ax.plot([x, x], extent[2:], color="red", lw=1.0, ls="--")
        ax.set_aspect("equal")
        ax.set_xlabel("x (m)")
        if letter == "A":
            ax.set_ylabel("y (m)")
        flopy.plot.styles.heading(ax=ax, letter=letter, heading=heading)
    fig.colorbar(
        pa,
        ax=list(axd.values()),
        orientation="horizontal",
        shrink=0.5,
        label="Interface elevation in the lower aquifer (m)",
    )

**What to look for.** Before pumping (A) the saltwater forms a wedge in the lower aquifer. Its toe lies 1.7 to 2.7 km from the coast in the west of the valley, farthest beneath the stream, and only 0.3 km in the east, where the freshwater heads are higher. At the coast the interface lies about -22 m in the west and -38 m in the east, and reaches the top of the lower aquifer, -4.6 m, beneath the mouth of the stream. The light strip near x = 1,300 m, where the interface is high far inland, lies beneath the stream, which lowers the heads around it. After 20 years of pumping (B) the toe has moved inland by up to 0.46 km in the middle and east of the valley, and the interface has risen around the `vc` and proposed wells. The `reilly` well in the north is far from any saltwater.

## The interface in cross section

Draw north-south cross sections through the `vc` well and the proposed well with `flopy.plot.PlotCrossSection()`. `plot_fill_between()` fills each cell up to the interface, clipped to the cell: light orange below the interface before pumping, dark orange for the saltwater that moved in during 20 years of pumping, and blue for the freshwater above the interface after pumping. The water table is drawn in blue, the land surface in brown, and the clay is drawn over the fill in dark brown. Thin gray lines mark the bottom of each layer.

In [ ]:
fresh_color, salt_color, clay_color = "#9ecae1", "#fdae6b", "#8c6d4f"
new_salt_color = "#e6550d"  # saltwater that moved in during pumping
clay = np.ma.masked_where(idomain != 1, np.ones(idomain.shape))
clay[[0, 2]] = np.ma.masked


with flopy.plot.styles.USGSPlot():
    fig, axd = plt.subplot_mosaic(
        [["A"], ["B"]], figsize=(width, 1.1 * width), layout="constrained", sharex=True
    )
    for letter, (well, j) in zip(axd, sections.items()):
        ax = axd[letter]
        xs = flopy.plot.PlotCrossSection(model=gwf, ax=ax, line={"column": j})
        # saltwater after pumping, then the part that was already there before
        xs.plot_fill_between(
            np.clip(zeta["after"], botm, tops),
            colors=(fresh_color, new_salt_color),
            lw=0,
        )
        xs.plot_fill_between(
            np.clip(np.minimum(zeta["before"], zeta["after"]), botm, tops),
            colors=("none", salt_color),
            lw=0,
        )
        # the unsaturated zone above the water table
        xs.plot_fill_between(
            np.clip(head["after"], botm, tops), colors=("white", "none"), lw=0
        )
        xs.plot_array(clay, cmap=mpl.colors.ListedColormap([clay_color]))
        for k in range(botm.shape[0]):
            xs.plot_surface(botm[k], color="0.3", lw=0.5)
        xs.plot_surface(top, color="saddlebrown", lw=1.0)
        xs.plot_surface(head["after"][0], color="blue", lw=1.0)
        i = wells[well][0]
        y = (i + 0.5) * gwf.dis.delc.array[i]
        ax.plot([y, y], [tops[2, i, j], botm[2, i, j]], color="purple", lw=3)
        ax.set_ylim(-100.0, 20.0)
        ax.set_ylabel("Elevation (m)")
        flopy.plot.styles.heading(
            ax=ax, letter=letter, heading=f"Section through the {well} well"
        )
    axd["B"].set_xlabel("Distance from the north edge (m)")
    handles = [
        mpl.patches.Patch(color=fresh_color, label="Freshwater after pumping"),
        mpl.patches.Patch(color=salt_color, label="Saltwater before pumping"),
        mpl.patches.Patch(color=new_salt_color, label="Saltwater moved in by pumping"),
        mpl.patches.Patch(color=clay_color, label="Clay"),
        mpl.lines.Line2D([], [], color="blue", label="Water table"),
        mpl.lines.Line2D([], [], color="purple", lw=3, label="Well screen"),
        mpl.lines.Line2D([], [], color="0.3", lw=0.5, label="Layer bottom"),
    ]
    flopy.plot.styles.graph_legend(
        ax=axd["B"],
        handles=handles,
        labels=[h.get_label() for h in handles],
        loc="upper center",
        bbox_to_anchor=(0.5, -0.2),
        ncol=3,
    )

**What to look for.** Before pumping (light orange) the interface lies at -49.5 m beneath the `vc` well (A), and there is no saltwater beneath the proposed well (B). After 20 years of pumping (dark orange) the interface has risen to -8.0 m and -9.1 m, less than 5 m below the top of the lower aquifer, so nearly the whole screen of each well is in saltwater. Where the clay is absent, nothing separates the upper aquifer from the rising saltwater, but the upper aquifer stays fresh except beneath the mouth of the stream.

## Are the wells drawing in seawater?

Read the flow-reduction files with pandas. Each row is a well whose rate was reduced in a time step: `rate-requested` is the rate in the well package and `rate-actual` is the rate MODFLOW pumped. Rates are negative for pumping. `boundnumber` is the position of the well in its package, counted from 1: the `vc` well is the second well of the `pwell` package, and the proposed well is the only well of the `prediction` package.

The time axis is years since pumping started at the end of the steady-state period.

In [ ]:
t_steady = head_file.get_times()[0]
afr = {
    "vc": ("pwell", 2),
    "proposed": ("prediction", 1),
}
rates = {}
for well, (pname, number) in afr.items():
    df = pd.read_csv(swi_ws / f"{name}.{pname}.afr.csv")
    df = df[df["boundnumber"] == number]
    df.index = (df["time"] - t_steady) / 365.25
    rates[well] = -df[["rate-requested", "rate-actual"]]
    last = rates[well].iloc[-1]
    print(
        f"{well}: requested {last['rate-requested']:.0f} m3/d, "
        f"pumped {last['rate-actual']:.0f} m3/d at the end "
        f"({last['rate-actual'] / last['rate-requested']:.0%})"
    )

In [ ]:
with flopy.plot.styles.USGSPlot():
    fig, ax = plt.subplots(figsize=(width, 0.4 * width), layout="constrained")
    for (well, df), color in zip(rates.items(), ("C0", "C1")):
        ax.plot(
            df.index,
            df["rate-requested"],
            color=color,
            ls="--",
            label=f"{well}, requested",
        )
        ax.plot(
            df.index,
            df["rate-actual"],
            color=color,
            label=f"{well}, pumped",
        )
    ax.set_xlim(0, 20)
    ax.set_ylim(0, 12000)
    flopy.plot.styles.xlabel(ax=ax, label="Years since pumping started")
    flopy.plot.styles.ylabel(ax=ax, label="Pumping rate (m³/d)")
    flopy.plot.styles.graph_legend(ax=ax, loc="upper left", ncol=2)

**What to look for.** The `vc` well keeps its full rate of 7,589 m³/d for two months, is cut back to 6,429 m³/d in the third month, falls to 4,048 m³/d (53 percent) by the end of its first year, and pumps 3,114 m³/d (41 percent) by the end of the simulation. The proposed well starts with no saltwater beneath it and keeps its full rate of 8,495 m³/d for a year and a half before the saltwater reaches it; it pumps 6,555 m³/d (77 percent) by the end. Neither well can pump its requested rate without drawing in seawater. The `reilly` well never appears in the file: it pumps its full rate throughout.

## How saltwater changes the water budget

Compare the water budget of the three-layer model without SWI and with it. As in [`synthetic-valley-a-base`](synthetic-valley-a-base.ipynb), read the budget from the listing file with `output.list()` and `get_dataframes()`, convert the rates to million cubic meters per year (million m³/yr), and give each term the same color as there.

The listing file reports the budget at the end of each stress period, so each year's rates are those at the end of the year, after its last monthly time step. With SWI the budget has one more term, `STORAGE`: the freshwater released as saltwater moves into a cell and the interface rises, or taken up as the interface falls.

In [ ]:
# m³/d to million m³/yr
M3D_TO_MCM_PER_YEAR = 365.25 / 1.0e6
runs_budget = {
    "Without SWI": fresh.get_model(name),
    "With SWI": gwf,
}
budgets = {}
for run, model in runs_budget.items():
    df, _ = model.output.list().get_dataframes(start_datetime="1962-01-01")
    budgets[run] = df * M3D_TO_MCM_PER_YEAR

# budget terms and colors of synthetic-valley-a-base, plus the drains and the
# SWI storage
terms = {
    "RCH": "Recharge",
    "RIV": "River",
    "GHB": "Coast",
    "EVT": "Lake evaporation",
    "STO-SY": "Storage (specific yield)",
    "STO-SS": "Storage (specific storage)",
    "WEL": "Pumping wells",
    "WEL2": "Proposed well",
    "DRN": "Seepage drains",
    "STORAGE": "Interface storage",
}
colors = dict(
    zip(
        terms,
        [
            "#2a78d6",
            "#eb6834",
            "#1baf7a",
            "#eda100",
            "#a8326e",
            "#e87ba4",
            "#008300",
            "#6250d6",
            "#7f7f7f",
            "#8c564b",
        ],
    )
)


def rate(df, key, side):
    # a term the run does not have, such as STORAGE without SWI, is zero
    column = f"{key}_{side}"
    return df[column] if column in df else pd.Series(0.0, index=df.index)


for run, df in budgets.items():
    summary = ", ".join(
        f"{label} {rate(df, key, 'OUT').iloc[0]:.2f}"
        for key, label in terms.items()
        if rate(df, key, "OUT").iloc[0] > 0
    )
    print(f"{run}, steady-state outflow: {summary} million m³/yr")

### Steady state

Draw where the recharge leaves at steady state, before pumping, as a pie chart for each run.

In [ ]:
with flopy.plot.styles.USGSPlot():
    fig, axd = plt.subplot_mosaic(
        [list(budgets)], figsize=(width, 0.45 * width), layout="constrained"
    )
    for run, df in budgets.items():
        outflow = {
            key: rate(df, key, "OUT").iloc[0]
            for key in terms
            if rate(df, key, "OUT").iloc[0] > 0
        }
        total = sum(outflow.values())
        ax = axd[run]
        ax.pie(
            outflow.values(),
            colors=[colors[key] for key in outflow],
            labels=[
                f"{terms[key]}\n{q:.2f} ({q / total:.0%})" for key, q in outflow.items()
            ],
            startangle=90,
            counterclock=False,
            wedgeprops={"edgecolor": "white", "linewidth": 1.5},
        )
        ax.set_aspect("equal")
        flopy.plot.styles.heading(ax=ax, heading=f"{run}, {total:.1f} million m³/yr")

### Transient periods

Draw the budget of each year as stacked bars, inflows upward and outflows downward, one panel for each run on the same scale.

In [ ]:
with flopy.plot.styles.USGSPlot():
    fig, axd = plt.subplot_mosaic(
        [[run] for run in budgets],
        figsize=(width, 1.1 * width),
        layout="constrained",
    )
    ylim = (
        np.ceil(
            max(
                df.iloc[1:][["TOTAL_IN", "TOTAL_OUT"]].to_numpy().max()
                for df in budgets.values()
            )
        )
        + 3.0
    )
    handles = {}
    for run, df in budgets.items():
        ax = axd[run]
        transient = df.iloc[1:]
        # each period ends on January 1, so it covers the year before its date
        bar_years = np.array([date.year - 1 for date in transient.index])
        for side, sign in (("IN", 1.0), ("OUT", -1.0)):
            bottom = np.zeros(len(bar_years))
            for key in terms:
                q = sign * rate(transient, key, side).to_numpy()
                if (q == 0).all():
                    continue
                bars = ax.bar(
                    bar_years,
                    q,
                    bottom=bottom,
                    width=0.8,
                    color=colors[key],
                    edgecolor="white",
                    linewidth=0.5,
                )
                handles.setdefault(key, bars)
                bottom += q
        ax.axhline(0.0, color="black", lw=0.75)
        ax.set_ylim(-ylim, ylim)
        ax.set_xticks(bar_years[::2])
        ax.yaxis.set_major_formatter(
            mpl.ticker.FuncFormatter(lambda y, pos: f"{abs(y):g}")
        )
        flopy.plot.styles.add_text(ax=ax, text="Inflow", x=0.01, y=0.97, va="top")
        flopy.plot.styles.add_text(ax=ax, text="Outflow", x=0.01, y=0.03, va="bottom")
        flopy.plot.styles.heading(
            ax=ax, heading=f"Water budget by year, {run[0].lower()}{run[1:]}"
        )
        flopy.plot.styles.ylabel(ax=ax, label="Rate (million m³/yr)")
    flopy.plot.styles.xlabel(ax=axd["With SWI"], label="Year")
    keys = [key for key in terms if key in handles]
    fig.legend(
        handles=[handles[key] for key in keys],
        labels=[terms[key] for key in keys],
        loc="outside lower center",
        ncol=3,
        frameon=False,
    )

In [ ]:
# the rates with SWI by the year each stress period covers, leaving out terms
# that are zero in every year
df = budgets["With SWI"]
table = pd.DataFrame(
    {f"{key}_{side}": rate(df, key, side) for side in ("IN", "OUT") for key in terms}
)
table = table.loc[:, (table != 0).any()]
table.index = ["Steady state"] + [date.year - 1 for date in table.index[1:]]
table.round(2)

**What to look for.** At steady state SWI moves 1.96 million m³/yr of the outflow from the coast to the river: the coast takes 24 percent of the recharge instead of 32 percent. Once pumping starts the two runs part ways. Without SWI the wells pump the 3.46 million m³/yr requested, and the proposed well another 3.10 million m³/yr from 2016, so the river drains 8 to 10.7 million m³/yr and the coast 4.3 to 6.8 million m³/yr. With SWI the `vc` and proposed wells are cut back to keep them from drawing in seawater: the pumping wells take 1.77 to 2.17 million m³/yr and the proposed well 2.19 to 3.10 million m³/yr, so the river still drains 11 to 14 million m³/yr and the coast 3.9 to 5.5 million m³/yr. Interface storage is the new term. It releases freshwater when the interface rises: 2.46 million m³/yr at the end of 2016, as the proposed well starts, and most of all 5.38 million m³/yr at the end of the dry year 2018. In the wetter years that follow, the interface falls back and the aquifer takes freshwater back into storage, up to 0.79 million m³/yr. In 2018 the river drains 10.7 million m³/yr with SWI but only 5.3 million m³/yr without it: in single-fluid mode the interface rises the moment the heads fall, and the freshwater it releases keeps the river flowing.

## Two-fluid SWI: how fast does the saltwater move?

Single-fluid SWI assumes the saltwater stands still and moves into place the moment the freshwater head changes: the interface follows the Ghyben–Herzberg relation at every time step. In fact the saltwater has to flow to get there. In **two-fluid** mode MODFLOW simulates the saltwater too, as a second groundwater flow model on the same grid. The SWI-SWI exchange couples the two models: each uses the head of the other to place the interface, and the saltwater flows through the saltwater zone of each cell.

Start from the single-fluid simulation, with its sea boundary, SWI Package, flow reduction, and monthly time steps, by loading it and pointing it at a new workspace.

In [ ]:
two_ws = pl.Path("models/synthetic-valley-swi-2fluid")
two = flopy.mf6.MFSimulation.load(sim_ws=swi_ws, verbosity_level=0)
two.set_sim_path(two_ws)
fresh_model = two.get_model(name)

### The saltwater model

The saltwater model uses the grid, storage, and initial heads of the freshwater model. Its hydraulic conductivity is that of the freshwater model times $1+\delta$: the same aquifer passes 2.5 percent more of the denser saltwater for the same head gradient. Its starting head is sea level everywhere.

The saltwater model has no recharge, stream, or wells. Its only boundary is the sea, on the coastal face of both aquifers, at a saltwater head equal to sea level. The saltwater enters the lower aquifer at the coast; in the freshwater model that face is closed, as before.

In [ ]:
salt_name = "svs"
salt = flopy.mf6.ModflowGwf(
    two, modelname=salt_name, newtonoptions="newton", save_flows=True
)
dis = fresh_model.dis
flopy.mf6.ModflowGwfdis(
    salt,
    length_units="meters",
    nlay=dis.nlay.data,
    nrow=dis.nrow.data,
    ncol=dis.ncol.data,
    delr=dis.delr.array,
    delc=dis.delc.array,
    top=dis.top.array,
    botm=dis.botm.array,
    idomain=dis.idomain.array,
)
flopy.mf6.ModflowGwfic(salt, strt=sea_level)
flopy.mf6.ModflowGwfnpf(
    salt,
    icelltype=0,
    k=fresh_model.npf.k.array * (1.0 + delta),
    k33=fresh_model.npf.k33.array,
    k33overk=True,
)
flopy.mf6.ModflowGwfsto(
    salt,
    iconvert=0,
    ss=fresh_model.sto.ss.array,
    sy=fresh_model.sto.sy.array,
    steady_state={0: True},
    transient={1: True},
)

# the sea on the coastal face of both aquifers, from the model without SWI
sea = fresh.get_model(name).ghb.stress_period_data.get_data(0)
flopy.mf6.ModflowGwfghb(
    salt,
    pname="sea",
    stress_period_data={
        0: [
            (cellid, sea_level, cond)
            for cellid, cond in zip(sea["cellid"], sea["cond"])
        ]
    },
)
flopy.mf6.ModflowGwfswi(
    salt,
    relative_density_difference=delta,
    zeta_filerecord=f"{salt_name}.zta",
)
flopy.mf6.ModflowGwfoc(
    salt,
    head_filerecord=f"{salt_name}.hds",
    budget_filerecord=f"{salt_name}.cbc",
    saverecord=[("head", "all"), ("budget", "all")],
)
two.model_names

### Couple the models and solve them together

`flopy.mf6.ModflowSwiswi()` adds the SWI-SWI exchange between the freshwater model (`exgmnamea`) and the saltwater model (`exgmnameb`). Both models must be solved in one solution, so replace the Iterative Model Solution (**IMS**) with one that holds both. The two models pull on each other through the interface, and **backtracking** keeps the outer iterations from overshooting: when an iteration increases the residual, MODFLOW shortens the head change and tries again.

In [ ]:
flopy.mf6.ModflowSwiswi(two, exgtype="SWI6-SWI6", exgmnamea=name, exgmnameb=salt_name)

two.remove_package(two.get_package("ims"))
ims = flopy.mf6.ModflowIms(
    two,
    print_option="summary",
    outer_dvclose=1.0e-4,
    outer_maximum=500,
    inner_maximum=100,
    inner_dvclose=1.0e-5,
    linear_acceleration="bicgstab",
    backtracking_number=20,
    backtracking_tolerance=1.05,
    backtracking_reduction_factor=0.1,
    backtracking_residual_limit=0.002,
)
two.register_ims_package(ims, [name, salt_name])

Write and run the two-fluid simulation. It solves two models and takes longer than the single-fluid run, about half a minute.

In [ ]:
two.write_simulation(silent=True)
success, buff = two.run_simulation(silent=True)
assert success, "MODFLOW 6 did not terminate normally"

### Check the steady state

Before pumping, nothing drives the saltwater, so it should come to rest at sea level and give the same interface as the single-fluid model.

In [ ]:
two_zeta_file = flopy.utils.HeadFile(two_ws / f"{name}.zta", text="zeta")
salt_head = flopy.utils.HeadFile(two_ws / f"{salt_name}.hds").get_data(idx=0)
d = two_zeta_file.get_data(idx=0) - zeta["before"]
print(
    f"saltwater head before pumping: {salt_head[idomain > 0].min():.4f} to "
    f"{salt_head[idomain > 0].max():.4f} m"
)
print(f"largest difference from the single-fluid interface: {np.abs(d).max():.3f} m")

### The interface beneath the wells

Read the interface beneath the `vc` and proposed wells at every time step of both runs, and the pumped rates from the flow-reduction files. The `vc` well is shown for its first three years of pumping, from the end of the steady-state period, and the proposed well for its first ten years, from year 10.

In [ ]:
single_zeta_file = zeta_file
times = np.array(single_zeta_file.get_times())
years = (times - t_steady) / 365.25
runs = {"single-fluid": swi_ws, "two-fluid": two_ws}
zeta_series = {
    run: flopy.utils.HeadFile(ws / f"{name}.zta", text="zeta").get_alldata()
    for run, ws in runs.items()
}
starts = {"vc": 0.0, "proposed": 10.0}  # years since pumping started
spans = {"vc": 3.0, "proposed": 10.0}  # years shown for each well


def pumped(ws, well):
    # the pumped rate at every time step after the well starts; the file lists
    # only the time steps in which the rate was reduced
    pname, number = afr[well]
    df = pd.read_csv(ws / f"{name}.{pname}.afr.csv")
    df = df[df["boundnumber"] == number]
    # times rounded to the day, so the two files' times match
    rate = pd.Series(-df["rate-actual"].values, index=df["time"].round(3).values)
    after = times[(years > starts[well]) & (years > 0.0)].round(3)
    requested = rates[well]["rate-requested"].iloc[0]
    rate = rate.reindex(after).fillna(requested)
    rate.index = (rate.index - t_steady) / 365.25
    return rate


with flopy.plot.styles.USGSPlot():
    fig, axd = plt.subplot_mosaic(
        [["A", "B"], ["C", "D"]],
        figsize=(width, 0.8 * width),
        layout="constrained",
    )
    for col, well in zip(("AC", "BD"), starts):
        i, j = wells[well]
        t0 = starts[well]
        for run, ls in zip(runs, ("--", "-")):
            axd[col[0]].plot(
                years, zeta_series[run][:, 2, i, j], color="black", ls=ls, label=run
            )
            rate = pumped(runs[run], well)
            axd[col[1]].plot(rate.index, rate, color="C0", ls=ls, label=run)
        requested = rates[well]["rate-requested"].iloc[0]
        axd[col[1]].axhline(requested, color="0.5", lw=0.75, label="requested")
        axd[col[0]].set_ylim(botm[2, i, j], tops[2, i, j])
        axd[col[1]].set_ylim(0, 1.1 * requested)
        axd[col[0]].set_xlim(t0, t0 + spans[well])
        axd[col[1]].set_xlim(t0, t0 + spans[well])
        flopy.plot.styles.heading(
            ax=axd[col[0]], letter=col[0], heading=f"Interface beneath the {well} well"
        )
        flopy.plot.styles.heading(
            ax=axd[col[1]], letter=col[1], heading=f"Pumping from the {well} well"
        )
        flopy.plot.styles.xlabel(ax=axd[col[1]], label="Years since pumping started")
    flopy.plot.styles.ylabel(ax=axd["A"], label="Elevation (m)")
    flopy.plot.styles.ylabel(ax=axd["C"], label="Pumping rate (m³/d)")
    # one legend below the panels, so it hides none of the curves
    handles = [
        mpl.lines.Line2D([], [], color="black", ls="--", label="Single-fluid"),
        mpl.lines.Line2D([], [], color="black", label="Two-fluid"),
        mpl.lines.Line2D([], [], color="0.5", lw=0.75, label="Requested rate"),
    ]
    fig.legend(handles=handles, loc="outside lower center", ncol=3, frameon=False)

**What to look for.** The saltwater takes time to reach the wells. There is no saltwater beneath the proposed well (B) when it starts. The single-fluid interface rises the moment the freshwater head falls: to -28.0 m within six months and -10.0 m within two years. The two-fluid interface rises only as fast as saltwater can flow in from the coast: -62.7 m after six months, -40.4 m after two years, and -10.7 m only after five years. The single-fluid well is cut back from its nineteenth month (D), the two-fluid well only after five years. After ten years the two runs nearly agree, 6,555 and 6,792 m³/d. Beneath the `vc` well (A and C), which starts with the interface closer to the well, the two runs differ only in the first months: the single-fluid well is cut back from its third month and the two-fluid well from its fifth. Single-fluid SWI answers whether a well will draw in seawater; two-fluid SWI also answers how soon.

## Explore the interface through time

The figure below draws the interface from either run at any output time. Use the controls to choose:

- **Time**, from the end of the steady-state period to the end of the simulation, in years since pumping started,
- **Run**, single-fluid or two-fluid; the interface of the other run is drawn as a dashed line in the cross section,
- **Layer**, the layer shown in the map, and
- **Section**, a row or a column, and its number, counted from 1. The section is the red line on the map.

The controls are made with ipywidgets. Each change redraws the figure; dragging the slider redraws it when you let go.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

head_series = {
    run: flopy.utils.HeadFile(ws / f"{name}.hds").get_alldata()
    for run, ws in runs.items()
}
layer_names = ["upper aquifer", "clay", "lower aquifer"]
nrow, ncol = gwf.dis.nrow.data, gwf.dis.ncol.data
zmin, zmax = np.floor(botm.min()), np.ceil(top.max())


def interface_in_layer(z, k):
    # the interface where it lies within layer k, NaN elsewhere
    inside = (z[k] > botm[k]) & (z[k] < tops[k]) & (idomain[k] > 0)
    return np.where(inside, z[k], np.nan)


def draw(itime, run, layer, direction, index):
    other = "two-fluid" if run == "single-fluid" else "single-fluid"
    z = zeta_series[run][itime]
    h = head_series[run][itime]
    line = {direction: index - 1}
    with flopy.plot.styles.USGSMap():
        fig, axd = plt.subplot_mosaic(
            [["map", "xs"]],
            figsize=(1.6 * width, 0.75 * width),
            layout="constrained",
            width_ratios=[1.0, 1.7],
        )
        ax = axd["map"]
        salt_zone = (z[layer] > botm[layer]) & (idomain[layer] > 0)
        mm = flopy.plot.PlotMapView(model=gwf, ax=ax, layer=layer)
        pa = mm.plot_array(
            np.ma.masked_where(~salt_zone, np.minimum(z[layer], tops[layer])),
            vmin=zmin,
            vmax=zmax,
        )
        if salt_zone.any():
            mm.contour_array(salt_zone.astype(float), levels=[0.5], colors="black")
        for well, (i, j) in wells.items():
            x, y = gwf.modelgrid.xcellcenters[i, j], gwf.modelgrid.ycellcenters[i, j]
            ax.plot(x, y, "o", ms=4, mfc="white", mec="black")
        if direction == "column":
            x = gwf.modelgrid.xcellcenters[0, index - 1]
            ax.plot([x, x], extent[2:], color="red", lw=1.0)
        else:
            y = gwf.modelgrid.ycellcenters[index - 1, 0]
            ax.plot(extent[:2], [y, y], color="red", lw=1.0)
        ax.set_aspect("equal")
        ax.set_xlabel("x (m)")
        ax.set_ylabel("y (m)")
        flopy.plot.styles.heading(
            ax=ax, heading=f"Layer {layer + 1}, {layer_names[layer]}"
        )
        fig.colorbar(
            pa, ax=ax, orientation="horizontal", shrink=0.8, label="Interface (m)"
        )

        ax = axd["xs"]
        xs = flopy.plot.PlotCrossSection(model=gwf, ax=ax, line=line)
        xs.plot_fill_between(
            np.clip(z, botm, tops), colors=(fresh_color, salt_color), lw=0
        )
        # the unsaturated zone above the water table
        xs.plot_fill_between(np.clip(h, botm, tops), colors=("white", "none"), lw=0)
        xs.plot_array(clay, cmap=mpl.colors.ListedColormap([clay_color]))
        for k in range(botm.shape[0]):
            xs.plot_surface(botm[k], color="0.3", lw=0.5)
        xs.plot_surface(top, color="saddlebrown", lw=1.0)
        xs.plot_surface(np.ma.masked_greater(h[0], 1.0e29), color="blue", lw=1.0)
        for k in (0, 2):
            xs.plot_surface(
                interface_in_layer(zeta_series[other][itime], k), color="black", ls="--"
            )
        ax.set_ylim(-100.0, 20.0)
        ax.set_xlabel(
            "Distance from the north edge (m)"
            if direction == "column"
            else "Distance from the west edge (m)"
        )
        ax.set_ylabel("Elevation (m)")
        flopy.plot.styles.heading(
            ax=ax,
            heading=f"{run.capitalize()}, {direction} {index}, "
            f"{years[itime]:.2f} years since pumping started",
        )
        handles = [
            mpl.patches.Patch(color=fresh_color, label="Freshwater"),
            mpl.patches.Patch(color=salt_color, label="Saltwater"),
            mpl.patches.Patch(color=clay_color, label="Clay"),
            mpl.lines.Line2D(
                [], [], color="black", ls="--", label=f"{other} interface"
            ),
            mpl.lines.Line2D([], [], color="blue", label="Water table"),
            mpl.lines.Line2D([], [], color="0.3", lw=0.5, label="Layer bottom"),
        ]
        flopy.plot.styles.graph_legend(
            ax=ax,
            handles=handles,
            labels=[hh.get_label() for hh in handles],
            loc="upper center",
            bbox_to_anchor=(0.5, -0.15),
            ncol=3,
        )
    plt.show()


time_slider = widgets.SelectionSlider(
    options=[(f"{y:.2f} years", n) for n, y in enumerate(years)],
    value=len(years) - 1,
    description="Time",
    continuous_update=False,
    layout=widgets.Layout(width="60%"),
)
run_buttons = widgets.ToggleButtons(
    options=list(runs), value="two-fluid", description="Run"
)
layer_menu = widgets.Dropdown(
    options=[(f"{k + 1}, {n}", k) for k, n in enumerate(layer_names)],
    value=2,
    description="Layer",
)
direction_buttons = widgets.RadioButtons(
    options=["column", "row"], value="column", description="Section"
)
index_slider = widgets.IntSlider(
    value=wells["proposed"][1] + 1,
    min=1,
    max=ncol,
    description="Number",
    continuous_update=False,
)


def set_index_range(change):
    # a column runs from 1 to ncol, a row from 1 to nrow
    i, j = wells["proposed"]
    if change["new"] == "column":
        index_slider.max, index_slider.value = ncol, j + 1
    else:
        index_slider.max, index_slider.value = nrow, i + 1


direction_buttons.observe(set_index_range, names="value")

out = widgets.interactive_output(
    draw,
    {
        "itime": time_slider,
        "run": run_buttons,
        "layer": layer_menu,
        "direction": direction_buttons,
        "index": index_slider,
    },
)
display(
    widgets.VBox(
        [
            time_slider,
            run_buttons,
            widgets.HBox([layer_menu, direction_buttons, index_slider]),
            out,
        ]
    )
)

**Things to try.** Set the time to the start and step it forward a month at a time with the arrow keys, first for the two-fluid run and then for the single-fluid run, along column 16 through the proposed well. Switch the section to row 35, which runs west to east through the proposed well. Then move the time to year 10, when the proposed well starts, and watch the interface beneath it rise.

## Recap

You combined the layers of the synthetic-valley base model into one layer per aquifer and checked the result against the base model, set the sea to an equivalent freshwater head on the upper aquifer, added the SWI Package, and let the wells cut back their pumping when the interface reached them. You mapped the saltwater wedge in the lower aquifer before and after 20 years of pumping, drew it in cross section, and read from the well output that the `vc` and proposed wells would draw in seawater at the rates requested, and compared the water budget with and without SWI. Finally, you added a saltwater model coupled to the freshwater model with the SWI-SWI exchange, found that it takes months, not moments, for the saltwater to rise beneath the wells, and explored the interface of both runs through time.